___

# Main Goal:
### Build and evaluate machine-learning models that classify text messages as spam or ham.

___

# Spam Message Detection Using Machine Learning

___

## 1. Project Overview

This project builds a machine-learning classifier that labels text messages as spam or ham.
The project compares multiple models, evaluates them using classification metrics,
analyzes common errors, and saves the best model for future predictions.

### Business / practical value

Spam filtering can reduce unwanted messages, scams, phishing attempts,
and time wasted by users. A good classifier should detect spam while avoiding
incorrectly marking legitimate messages as spam.

## 2. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Load the Dataset

In [ ]:
df = pd.read_csv("../data/SMSSpamCollection",
                sep="\t",
                header=None,
                names=["label", "message"])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

___
The dataset contains one target column (`label`) and one text feature (`message`).
The target has two categories: spam and ham.
___

## 4. Data Understanding and Cleaning

In [ ]:
df["label"].value_counts(normalize=True) * 100

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

___
Duplicate messages may represent repeated real-world spam campaigns.
However, exact duplicates can also cause overly optimistic evaluation if the same
message appears in both training and testing data.
___

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.duplicated().sum(), df.shape

___
**Duplicates are removed to reduce leakage risk.**
___

In [ ]:
label_map = {"ham": 0, "spam": 1}
df["target"] = df["label"].map(label_map)

In [ ]:
df.head()

In [ ]:
df.isna().sum()

## 5. Exploratory Data Analysis

In [ ]:
sns.countplot(x=df["label"])
plt.show;

___
The dataset has more ham messages than spam messages. Therefore, accuracy alone
is not enough: a model could achieve a high accuracy by predicting ham too often.
Precision, recall, and F1-score for the spam class are important.
___

In [ ]:
df["message_length"] = df["message"].apply(len)
df["word_count"] = df["message"].apply(lambda x: len(x.split()))

In [ ]:
df.head()

In [ ]:
sns.kdeplot(df[df["label"]=="ham"]["message_length"], fill=True, color="blue", label="Ham")
sns.kdeplot(df[df["label"]=="spam"]["message_length"], fill=True, color="red", label="Spam")

plt.title("KDE Plot of Message Length by Message Type")
plt.xlabel("Message Length (characters)")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
sns.boxplot(x="label",
            y="message_length",
            data=df)

plt.title("Box Plot of Message Length by Message Type")
plt.xlabel("Message Type (ham/spam)")
plt.ylabel("Message Length (characters)")
plt.show()

___
The plots show that spam messages have higher median length than ham, Ham messages often have extreme outliers, Spam has fewer but still some long outliers,
___

In [ ]:
merged_string = df.groupby("label")["message"].agg(" ".join)
ham_string = merged_string["ham"]
spam_string = merged_string["spam"]

In [ ]:
from wordcloud import WordCloud

ham_wordcloud = WordCloud(width=800,
                          height=400,
                          background_color="white").generate(ham_string)
spam_wordcloud = WordCloud(width=800,
                           height=400,
                           background_color="white").generate(spam_string)

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(ham_wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()
# word-cloud for ham words

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(spam_wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()
# word-cloud for spam words

## 6. Text Preprocessing

In [ ]:
import re
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

In [ ]:
STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', 'URL', text)
    text = re.sub(r'\d{10,}', 'PHONE', text)
    text = re.sub(r'[^\w\s!$]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in STOPWORDS]
    tagged_words = pos_tag(words)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in tagged_words]
    
    return " ".join(lemmatized)

In [ ]:
df["cleaned_message"] = df["message"].apply(lambda x: clean_text(x))

In [ ]:
df.sample(frac=1).head()

## 7. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["cleaned_message"]
Y = df["target"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X,
                                                    Y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=Y)

___
Stratification preserves the class distribution across training and test sets.
This is especially important because spam messages are less common than ham messages.
___

## 8. Baseline Model

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
model = Pipeline([
    ("vectorizer", CountVectorizer(stop_words="english")),
    ("classifier", MultinomialNB())
])

model.fit(X_train, Y_train)

In [ ]:
model.score(X_test, Y_test)

In [ ]:
Y_pred = model.predict(X_test)

In [ ]:
class_report = classification_report(Y_test, Y_pred)
print(class_report)

In [ ]:
conf_mat = confusion_matrix(Y_test, Y_pred)
print(conf_mat)

In [ ]:
pd.crosstab(Y_test,
            Y_pred,
            rownames=["True Label"],
            colnames=["Predicted Label"])

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(Y_test, Y_pred);

## 9. TF-IDF Feature Engineering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [ ]:
clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=5,
        max_df=0.9,
        stop_words="english",
        sublinear_tf=True,
    )),
    ("nb", MultinomialNB())
])

In [ ]:
clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

In [ ]:
logreg = Pipeline([
    ("tfifg", TfidfVectorizer(
        min_df=5,
        max_df=0.9,
        stop_words="english",
        sublinear_tf=True
    )),
    ("logreg", LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [ ]:
logreg.fit(X_train, Y_train)
logreg.score(X_test, Y_test)

In [ ]:
svc = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svc", LinearSVC())
])

In [ ]:
svc.fit(X_train, Y_train)
svc.score(X_test, Y_test)

## 10. Model Comparison

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

**Accuracy**

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
Count_NB_score = cross_val_score(model, X, Y, cv=kf, scoring="accuracy")
Count_NB_score

In [ ]:
Tfidf_NB_score = cross_val_score(clf, X, Y, cv=kf, scoring="accuracy")
Tfidf_NB_score

In [ ]:
Tfidf_logreg_score = cross_val_score(logreg, X, Y, cv=kf, scoring="accuracy")
Tfidf_logreg_score

In [ ]:
Tfidf_svc_score = cross_val_score(svc, X, Y, cv=kf, scoring="accuracy")
Tfidf_svc_score

**Classification Report**

In [ ]:
# Prediction for each model
Y1_pred = model.predict(X_test)
Y2_pred = clf.predict(X_test)
Y3_pred = logreg.predict(X_test)
Y4_pred = svc.predict(X_test)

In [ ]:
# Report for each model
Count_NB_report = classification_report(Y_test, Y1_pred, output_dict=True)
Tfidf_NB_report = classification_report(Y_test, Y2_pred, output_dict=True)
Tfidf_logred_report = classification_report(Y_test, Y3_pred, output_dict=True)
Tfidf_svc_report = classification_report(Y_test, Y4_pred, output_dict=True)

In [ ]:
import pandas as pd

data = {
    "Count NB": [
        Count_NB_report["weighted avg"]["precision"],
        Count_NB_report["weighted avg"]["recall"],
        Count_NB_report["weighted avg"]["f1-score"]
    ],
    "Tfidf NB": [
        Tfidf_NB_report["weighted avg"]["precision"],
        Tfidf_NB_report["weighted avg"]["recall"],
        Tfidf_NB_report["weighted avg"]["f1-score"]
    ],
    "Tfidf logreg": [
        Tfidf_logred_report["weighted avg"]["precision"],
        Tfidf_logred_report["weighted avg"]["recall"],
        Tfidf_logred_report["weighted avg"]["f1-score"]
    ],
    "Tfidf SVC": [
        Tfidf_svc_report["weighted avg"]["precision"],
        Tfidf_svc_report["weighted avg"]["recall"],
        Tfidf_svc_report["weighted avg"]["f1-score"]
    ]
}

report_df = pd.DataFrame(data, index=["Precision","Recall","F1-score"])
report_df

In [ ]:
report_df.plot(kind="bar", figsize=(10, 6))
plt.title("Model Comparison by Metrics")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(title="Models", bbox_to_anchor=(1.005, 1), loc="upper left")
plt.show()

___
**After comparing the four models, the LinearSVC model(with TfidfVectorizer) is chosen as the final model.**
___

## 11. Model Evaluation(for the final model)

In [ ]:
print(classification_report(Y_test, Y4_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(Y_test, Y4_pred);

## 12. Error Analysis

In [ ]:
error_analysis_df = X_test.to_frame(name="Message")
error_analysis_df["Predicted_label"] = Y4_pred
error_analysis_df["Actual_label"] = Y_test
error_analysis_df["Correctness"] = error_analysis_df.apply(
    lambda row: "✅" if row["Predicted_label"] == row["Actual_label"] else "❌",
    axis=1
)

In [ ]:
error_analysis_df.sample(frac=1).head()

## 13. Model Saving

In [ ]:
import joblib

In [ ]:
joblib.dump(svc, "../models/LinearSVC_TFIDF")

## 14. Limitations and Future Improvement

### Limitations
- The dataset may not represent current scam language.
- The classes are imbalanced.
- Text-only classification cannot detect all phishing attempts.
- A model trained on English messages may not work well for Amharic or other languages.

### Future Improvement
- Build a Streamlit web interface.